# Design Spaces 

## Setup

### Import Packages and Directories

In [ ]:
import flowtangent as ft
import jax
import jax.numpy as jnp
import numpy as np

from pathlib import Path

from flowtangent.models.kernels import RBFKernel, Matern52Kernel
from flowtangent.models.surrogates import SVGPSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough, ScalingManifold
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

import jax.numpy as jnp

In [2]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name: str):
    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(batch: dict):
    names = batch["foil_name"]
    
    # 1. Strip Xarray metadata if it slipped through
    if hasattr(names, "values"):
        names = names.values
        
    # 2. Force it into a flat, pure NumPy array
    names = np.asarray(names).flatten()
    
    # 3. Now iteration yields pure strings (or numpy.str_), which str() handles perfectly
    surfaces = [surf_lookup(str(name)) for name in names]
    
    batch["y_af"] = jnp.stack(surfaces)
    return batch

### Import Dataset

In [3]:
data_dir = Path("/home/jordan/dev/datasets/XFOIL_4D.zarr")
data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="jtsmart/XFOIL-4D", repo_type="dataset", filename="XFOIL_4D.zarr.zip")

if not (data_dir / 'zarr.json').exists():
    print("Extracting XFOIL 4D Dataset")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Extraction Complete.")
else:
    print ("Stored dataset detected.")

print(f"Dataset ready at: {data_dir}")

Stored dataset detected.
Dataset ready at: /home/jordan/dev/datasets/XFOIL_4D.zarr


In [5]:
XFD_arr = ft.utils.data.XArrayDataset(data_dir)

In [6]:
XFD_arr.full_vars

['Area',
 'Bot_Itr',
 'Bot_Xtr',
 'CD',
 'CDp',
 'CL',
 'CM',
 'J',
 'J/t',
 'Mach',
 'Re',
 'Slen',
 'Top_Itr',
 'Top_Xtr',
 'alpha',
 'bend_data',
 'conditions',
 'flap_angle',
 'flap_x/c',
 'foil_name',
 'polar_data',
 'skin_X_Iyy/t',
 'skin_X_Iyy/t(X-Xc)',
 'skin_X_Xc',
 'skin_X_max X-Xc',
 'skin_X_min X-Xc',
 'skin_Y_Ixx/t',
 'skin_Y_Ixx/t(Y-Yc)',
 'skin_Y_Yc',
 'skin_Y_max Y-Yc',
 'skin_Y_min Y-Yc',
 'solid_X_Iyy',
 'solid_X_Iyy/(X-Xc)',
 'solid_X_Xc',
 'solid_X_max X-Xc',
 'solid_X_min X-Xc',
 'solid_Y_Ixx',
 'solid_Y_Ixx/(Y-Yc)',
 'solid_Y_Yc',
 'solid_Y_max Y-Yc',
 'solid_Y_min Y-Yc']

In [7]:
XFD_raw = ft.utils.data.XArrayDataset(data_dir).to_tabular(["run", "alpha"])
XFD = XFD_raw.dropna(["alpha"])
XFD = XFD.transform_variable(name2surf)

Dropping NaNs from dataset:   0%|          | 0/73 [00:00<?, ?it/s]

In [8]:
unique_names = [str(n) for n in set(np.asarray(XFD_raw.foil_name))][1:]
surfs = [jnp.atleast_2d(surf_lookup(n)) for n in unique_names]
X_SURF = jnp.concatenate(surfs)

### Set up Parameters

In [9]:
# Inputs
y_af = Parameter(name="y_af", is_transform=True, transform_sources=("foil_name",))
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Parameter(name="alpha")

# Outputs
CL = Parameter(name="CL", is_output=True)
CD = Parameter(name="CD", is_output=True)
CM = Parameter(name="CM", is_output=True)

theta = (y_af, Mach, Re, flap_loc, flap_angle, alpha, CL, CD, CM)

### Set up Manifold

In [10]:
geofold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
geofold = geofold.fit(X_SURF)

machfold    = ScalingManifold(method="softplus")
refold      = ScalingManifold(method="logarithmic")
hingefold   = ScalingManifold(method="algebraic_bounded", bounds_min=0.6, bounds_max=0.85)
deltafold   = ScalingManifold(method="linear", scale_factors=10)
alphafold   = ScalingManifold(method="linear", scale_factors=15)

mfold = geofold + machfold + refold + hingefold + deltafold + alphafold

### Set up Surrogate

### Setup Design Space

In [ ]:
batch_size = 8192
beta = 0.5 * batch_size / len(XFD)

krnl = RBFKernel(input_dim=21)
surr = SVGPSurrogate(
    jax.random.key(42),
    input_dim=21,
    output_dim=3,
    kernel=krnl,
    num_inducing=1024,
    complexity_weight=beta
)

airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
)

In [ ]:
airspace = airspace.fit_models(batch_size=batch_size, surrogate_epochs=5)

In [21]:
eval = airspace.evaluate(airspace.project(XFD[0]))

In [22]:
eval.means

Array([ 0.42313131,  0.02062135, -0.09983778], dtype=float64)

In [31]:
evals = [airspace.evaluate(airspace.project(XFD[i])) for i in range(5)]

In [40]:
jnp.sum(airspace.surrogate._variational_mean ** 2)

Array(1901.85377024, dtype=float64)

In [37]:
for i, e in enumerate(evals):
    print(f"CL: {e.means[0]:.3f} ({e.variances[0]:.3f}) [{XFD[i]["CL"]:.3f}]")
    print(f"CD: {e.means[1]:.3f} ({e.variances[1]:.3f}) [{XFD[i]["CD"]:.3f}]")
    print(f"CM: {e.means[2]:.3f} ({e.variances[2]:.3f}) [{XFD[i]["CM"]:.3f}]\n")

CL: 0.423 (0.033) [0.596]
CD: 0.021 (0.001) [0.036]
CM: -0.100 (0.001) [-0.131]

CL: 0.563 (0.033) [0.729]
CD: 0.021 (0.001) [0.033]
CM: -0.105 (0.001) [-0.130]

CL: 0.696 (0.033) [0.841]
CD: 0.023 (0.001) [0.030]
CM: -0.108 (0.001) [-0.125]

CL: 0.817 (0.033) [0.944]
CD: 0.026 (0.001) [0.034]
CM: -0.110 (0.001) [-0.123]

CL: 0.924 (0.033) [1.021]
CD: 0.029 (0.001) [0.042]
CM: -0.109 (0.001) [-0.121]

